In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=10000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([8000, 10]) torch.Size([8000, 1])
torch.Size([2000, 10]) torch.Size([2000, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = optim.SGD(model.parameters(), lr=2e-2)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.28140273690223694
epoch:  1, loss: 0.14056310057640076
epoch:  2, loss: 0.07799132168292999
epoch:  3, loss: 0.05094663426280022
epoch:  4, loss: 0.039589185267686844
epoch:  5, loss: 0.0349295437335968
epoch:  6, loss: 0.03304855525493622
epoch:  7, loss: 0.03229620307683945
epoch:  8, loss: 0.031995952129364014
epoch:  9, loss: 0.031875330954790115
epoch:  10, loss: 0.031825847923755646
epoch:  11, loss: 0.03180455043911934
epoch:  12, loss: 0.031794484704732895
epoch:  13, loss: 0.03178894519805908
epoch:  14, loss: 0.031785257160663605
epoch:  15, loss: 0.031782347708940506
epoch:  16, loss: 0.03177977725863457
epoch:  17, loss: 0.031777359545230865
epoch:  18, loss: 0.03177502006292343
epoch:  19, loss: 0.03177272528409958
epoch:  20, loss: 0.03177045285701752
epoch:  21, loss: 0.03176819533109665
epoch:  22, loss: 0.031765952706336975
epoch:  23, loss: 0.031763706356287
epoch:  24, loss: 0.03176146745681763
epoch:  25, loss: 0.03175925090909004
epoch:  26, loss

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6794336438179016
Test metrics:  R2 = 0.6503345966339111


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = optim.Adam(model.parameters(), lr=1e-2)

all_loss = {}
for epoch in range(100):
    print("epoch: ", epoch, end="")
    all_loss[epoch + 1] = 0
    for batch_idx, (b_x, b_y) in enumerate(data_loader):
        pre = model(b_x)
        loss = loss_fn(pre, b_y)
        opt.zero_grad()
        loss.backward()

        # parameter update step based on optimizer
        opt.step()

        all_loss[epoch + 1] += loss
    all_loss[epoch + 1] /= len(data_loader)
    print(", loss: {}".format(all_loss[epoch + 1].cpu().detach().cpu().numpy().item()))

epoch:  0, loss: 0.12358660995960236
epoch:  1, loss: 0.04194790497422218
epoch:  2, loss: 0.03760319948196411
epoch:  3, loss: 0.03240930661559105
epoch:  4, loss: 0.030250441282987595
epoch:  5, loss: 0.029570244252681732
epoch:  6, loss: 0.02669578418135643
epoch:  7, loss: 0.022090859711170197
epoch:  8, loss: 0.015347696840763092
epoch:  9, loss: 0.01012551411986351
epoch:  10, loss: 0.008571026846766472
epoch:  11, loss: 0.007878920063376427
epoch:  12, loss: 0.007423748262226582
epoch:  13, loss: 0.0072967736050486565
epoch:  14, loss: 0.007246038410812616
epoch:  15, loss: 0.007150708232074976
epoch:  16, loss: 0.007064030971378088
epoch:  17, loss: 0.006978876888751984
epoch:  18, loss: 0.006899425759911537
epoch:  19, loss: 0.006809909362345934
epoch:  20, loss: 0.006725267972797155
epoch:  21, loss: 0.006633325479924679
epoch:  22, loss: 0.006541792768985033
epoch:  23, loss: 0.0064520156010985374
epoch:  24, loss: 0.006359562743455172
epoch:  25, loss: 0.006275370717048645


In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9973690509796143
Test metrics:  R2 = 0.997219979763031
